<a href="https://colab.research.google.com/github/youma-code/qqq/blob/main/1.3.0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
import time

# ======================
# データ取得
# ======================
ticker = "AAPL"
start_date = "2020-01-01"
end_date = pd.to_datetime("today").strftime("%Y-%m-%d")

def download_data(ticker, start, end):
    for _ in range(5):
        df = yf.download(ticker, start=start, end=end)
        if not df.empty:
            return df
        time.sleep(3)
    return pd.DataFrame()

df = download_data(ticker, start_date, end_date)

# ======================
# 整形
# ======================
if isinstance(df.columns, pd.MultiIndex):
    df = df[['Close', 'High', 'Low', 'Volume']].droplevel(1, axis=1)
else:
    df = df[['Close', 'High', 'Low', 'Volume']].copy()

df.columns = ['Close', 'High', 'Low', 'Volume']
df.dropna(inplace=True)

# ======================
# 特徴量作成（強化版）
# ======================

# --- SMA系 ---
df["SMA_5"] = df["Close"].rolling(5).mean()
df["SMA_25"] = df["Close"].rolling(25).mean()
df["SMA_50"] = df["Close"].rolling(50).mean()
df["SMA_200"] = df["Close"].rolling(200).mean()
df["SMA_DIFF"] = df["SMA_5"] - df["SMA_25"]
df["SMA_RATIO"] = df["SMA_5"] / (df["SMA_25"] + 1e-9)

# --- Target ---
future_max = df["Close"].shift(-1).rolling(3).max()
df["Target"] = ((future_max / df["Close"] - 1) > 0.0075).astype(int)

# --- RSI ---
delta = df["Close"].diff()
gain = delta.clip(lower=0).rolling(14).mean()
loss = (-delta.clip(upper=0)).rolling(14).mean()
rs = gain / (loss + 1e-9)
df["RSI"] = 100 - (100 / (1 + rs))

# --- MACD ---
ema12 = df["Close"].ewm(span=12).mean()
ema26 = df["Close"].ewm(span=26).mean()
df["MACD"] = ema12 - ema26
df["Signal_Line"] = df["MACD"].ewm(span=9).mean()
df["MACD_Hist"] = df["MACD"] - df["Signal_Line"]

# --- ボラ・リターン ---
df["Daily_Return"] = df["Close"].pct_change()
df["Volatility_Short"] = df["Daily_Return"].rolling(5).std()

# --- ATR ---
tr = pd.concat([
    df["High"] - df["Low"],
    abs(df["High"] - df["Close"].shift()),
    abs(df["Low"] - df["Close"].shift())
], axis=1).max(axis=1)

df["ATR"] = tr.rolling(14).mean()

# ======================
# 🔥追加強化特徴量
# ======================

# モメンタム
df["RET_3"] = df["Close"].pct_change(3)
df["RET_5"] = df["Close"].pct_change(5)
df["RET_10"] = df["Close"].pct_change(10)

# ボラ構造
df["VOL_10"] = df["Daily_Return"].rolling(10).std()
df["VOL_20"] = df["Daily_Return"].rolling(20).std()
df["VOL_RATIO"] = df["VOL_10"] / (df["VOL_20"] + 1e-9)

# レンジ度
df["RANGE_SCORE"] = abs(df["RSI"] - 50)

# モメンタム補助
df["MOMENTUM"] = df["Close"] - df["Close"].shift(5)

# ======================
# cleanup
# ======================
df = df.dropna()

# ======================
# ADX（ここが重要）
# ======================
window = 14

plus_dm = df["High"].diff()
minus_dm = -df["Low"].diff()

plus_dm = plus_dm.clip(lower=0)
minus_dm = minus_dm.clip(lower=0)

tr = pd.concat([
    df["High"] - df["Low"],
    (df["High"] - df["Close"].shift()).abs(),
    (df["Low"] - df["Close"].shift()).abs()
], axis=1).max(axis=1)

atr = tr.rolling(window).mean()

plus_di = 100 * (plus_dm.rolling(window).mean() / (atr + 1e-9))
minus_di = 100 * (minus_dm.rolling(window).mean() / (atr + 1e-9))

dx = (abs(plus_di - minus_di) / (plus_di + minus_di + 1e-9)) * 100

df["ADX"] = dx.rolling(window).mean()

# ======================
# 追加特徴量（ADX後にやる！）
# ======================

df["RET_3"] = df["Close"].pct_change(3)
df["RET_5"] = df["Close"].pct_change(5)
df["RET_10"] = df["Close"].pct_change(10)

df["VOL_10"] = df["Daily_Return"].rolling(10).std()
df["VOL_20"] = df["Daily_Return"].rolling(20).std()
df["VOL_RATIO"] = df["VOL_10"] / (df["VOL_20"] + 1e-9)

df["RANGE_SCORE"] = abs(df["RSI"] - 50)
df["MOMENTUM"] = df["Close"] - df["Close"].shift(5)

# ★ここで初めて作る
df["TREND_STRENGTH"] = df["ADX"] * df["MACD_Hist"]

# --- BB ---
ma = df["Close"].rolling(20).mean()
std = df["Close"].rolling(20).std()

df["Upper_BB"] = ma + 2 * std
df["Lower_BB"] = ma - 2 * std
df["BB_Width"] = (df["Upper_BB"] - df["Lower_BB"]) / (ma + 1e-9)

# --- lag ---
for lag in range(1, 6):
    df[f"Close_Lag_{lag}"] = df["Close"].shift(lag)

# ======================
# 🔥新規追加: レジーム分離と正規化ADX、市場レジーム
# ======================
df["ADX_normalized"] = df["ADX"] / 100  # ADXを0-1に正規化
# 初期データ準備ではデフォルトのADX閾値を使用 (Optunaで最適化)
default_adx_regime_threshold = 20
df["trend_regime"] = (df["ADX"] > default_adx_regime_threshold).astype(int)

# --- Market Regime ---
df["Market_Regime"] = "sideways" # Default
df.loc[df["SMA_50"] > df["SMA_200"], "Market_Regime"] = "uptrend"
df.loc[df["SMA_50"] < df["SMA_200"], "Market_Regime"] = "downtrend"

# ======================
# Market Regime One-Hot Encoding (Removed for string unification)
# ======================
df = pd.get_dummies(df, columns=["Market_Regime"])

# Ensure all three regime columns exist (Fixed as per user request)
# These lines are no longer needed if not one-hot encoding, but they are benign if Market_Regime is not a column
# for col in ["Market_Regime_uptrend", "Market_Regime_downtrend", "Market_Regime_sideways"]:
#     if col not in df.columns:
#         df[col] = 0

# ======================
# cleanup
# ======================
df.dropna(inplace=True)

print("完了:", df.shape)

/tmp/ipykernel_59200/1450734073.py:15: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start=start, end=end)
[*********************100%***********************]  1 of 1 completed

完了: (1365, 40)


In [2]:
# ======================
# 特徴量リスト（固定）
# ======================

features = [
    'SMA_5', 'SMA_25', 'SMA_DIFF', 'SMA_RATIO',
    'RSI', 'MACD', 'Signal_Line', 'MACD_Hist',
    'ATR',
    'Upper_BB', 'Lower_BB', 'BB_Width',
    'Volatility_Short',
    'ADX',
    'ADX_normalized',  # ←追加: 正規化ADX
    'trend_regime',
    'RET_5',
    'TREND_STRENGTH' # Missing feature added here
]
# ラグ特徴量（セル1で作ったやつに対応）
for lag in range(1, 6):
    features.append(f'Close_Lag_{lag}')

# 存在しない列があったら弾く（安全対策）
if 'df' not in globals():
    print("エラー: DataFrame 'df' が定義されていません。このセルを実行する前に、データをロードしてdfを作成する前のセルを実行してください。")
    raise NameError("DataFrame 'df' is not defined. Please run the data preparation cells above.")

# NOTE: Market_Regime will be one-hot encoded later on the full df.
# The encoded columns will be added to the features list dynamically.

features = [f for f in features if f in df.columns]

print("使用特徴量数:", len(features))
print(features[:10])

使用特徴量数: 23
['SMA_5', 'SMA_25', 'SMA_DIFF', 'SMA_RATIO', 'RSI', 'MACD', 'Signal_Line', 'MACD_Hist', 'ATR', 'Upper_BB']


In [3]:
# ======================
# X, y 作成
# ======================

# 念のためチェック
assert 'Target' in df.columns, "Target列が存在しません"

# Market_Regime はすでに前のセルでワンホットエンコーディング済み
df_encoded = df.copy() # df_encoded は df と同じデータフレームを指す

# One-hot encodedされたMarket_Regimeの列名を動的に取得
# ユーザーのリクエストに従い、すべてのMarket_Regime関連列を取得する
encoded_market_regime_cols = [col for col in df_encoded.columns if col.startswith('Market_Regime_')]

# ② feature_columnsは既存の特徴量とエンコードされたMarket_Regimeの列を結合
final_features = features + encoded_market_regime_cols

# Xとyをこのエンコード済みデータから作る
X = df_encoded[final_features].copy()
y = df_encoded['Target'].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Target分布:")
print(y.value_counts(normalize=True))

X shape: (1365, 25)
y shape: (1365,)
Target分布:
Target
0    0.528938
1    0.471062
Name: proportion, dtype: float64


In [4]:
# This cell is superseded by the logic in cauDYW8ZZ2xR, where X and y are created
# from the one-hot encoded DataFrame (df_encoded) using final_features.
# Leaving this as a placeholder to indicate it's no longer used.
# Original content:
X = df[features]
y = df['Target']
print(f"X and y defined. Feature matrix shape: {X.shape}")

X and y defined. Feature matrix shape: (1365, 23)


In [5]:
import numpy as np
import pandas as pd

def run_backtest(
    df,
    X_test,
    signal_multipliers,
    df_trade_info, # New parameter to pass df_test_slice with entry_score and proba
    initial_balance=100000,
    atr_tp_multiplier=2.5, # New: Parameter for Take Profit multiplier
    atr_sl_multiplier=1.2, # New: Parameter for Stop Loss multiplier
    trail_stop=False,      # New: Flag for trailing stop
    trail_ratio=0.5,       # New: Trailing stop ratio
    min_hold_bars=1        # New: Minimum bars to hold a position
):

    # --- テスト期間 ---
    df_bt = df.loc[X_test.index].copy()

    # 必須データチェック
    required_cols = ['Close', 'ATR']
    df_bt = df_bt.dropna(subset=required_cols)

    # signal_multipliers align
    signal_multipliers = signal_multipliers.loc[df_bt.index] # Align multipliers
    df_trade_info = df_trade_info.loc[df_bt.index] # Align df_trade_info

    # --- 初期化 ---
    balance = initial_balance
    equity_curve = []

    position = 0
    entry_price = 0
    entry_atr = 0
    current_stop_price = 0 # To track dynamically adjusted stop price
    current_take_profit_price = 0 # To track dynamically adjusted take profit price
    bars_held = 0 # New: Counter for bars held

    trade_count = 0
    win_count = 0
    trade_pnls = []
    trade_log = [] # New: To store detailed trade information

    # ====================== #
    # メインループ
    # ====================== #
    for i in range(len(df_bt) - 1):

        row = df_bt.iloc[i]
        next_row = df_bt.iloc[i + 1]
        current_price = next_row['Close'] # Ensure current_price is always defined for the current bar

        current_signal_multiplier = signal_multipliers.iloc[i]

        # ====================== #
        # エントリー
        # ====================== #
        if position == 0 and current_signal_multiplier > 0:

            atr = row['ATR']
            if pd.isna(atr) or atr == 0:
                equity_curve.append(balance)
                continue

            risk_amount = balance * 0.01 # Fixed risk_per_trade for now
            stop_distance = atr * atr_sl_multiplier

            unit_position_size = risk_amount / stop_distance

            entry_price = next_row['Close']
            entry_atr = atr # ATR at the time of entry

            # Capture entry_score and proba at the time of entry
            proba_at_trade = df_trade_info.iloc[i]['proba']
            entry_score = df_trade_info.iloc[i]['entry_score']
            current_regime = df_trade_info.iloc[i]['Market_Regime']

            position = unit_position_size * current_signal_multiplier
            trade_count += 1

            # Initial TP/SL for the trade
            current_stop_price = entry_price - (entry_atr * atr_sl_multiplier)
            current_take_profit_price = entry_price + (entry_atr * atr_tp_multiplier)
            bars_held = 0 # Reset bars held for new trade

        # ====================== #
        # ポジション管理
        # ====================== #
        elif position > 0:

            bars_held += 1
            # current_price = next_row['Close'] # This line is moved to the beginning of the loop
            exit_flag = False

            # Trailing Stop Logic (Apply only if position is profitable)
            if trail_stop and (current_price - entry_price) * position > 0: # Check if trade is currently profitable
                # Use the current row's ATR for trailing stop calculation
                trailing_stop_candidate = current_price - (row['ATR'] * trail_ratio)
                current_stop_price = max(current_stop_price, trailing_stop_candidate)

            # Check for Stop Loss (always immediate exit)
            if current_price <= current_stop_price:
                exit_flag = True
            # Check for Take Profit (respect min_hold_bars)
            elif current_price >= current_take_profit_price:
                if bars_held >= min_hold_bars:
                    exit_flag = True
                # Else, hold position if profitable but min_hold_bars not met

            if exit_flag:

                pnl = (current_price - entry_price) * position
                balance += pnl
                trade_pnls.append(pnl)

                win_flag = pnl > 0
                if win_flag:
                    win_count += 1

                trade_log.append({
                    'pnl': pnl,
                    'win': win_flag,
                    'proba': proba_at_trade,
                    'market_regime': current_regime,
                    'entry_score': entry_score
                })

                position = 0
                current_stop_price = 0 # Reset
                current_take_profit_price = 0 # Reset

        # ====================== #
        # equity
        # ====================== #
        if position > 0:
            unrealized = (current_price - entry_price) * position
            equity_curve.append(balance + unrealized)
        else:
            equity_curve.append(balance)

    # ====================== #
    # 強制決済
    # ====================== #
    if position > 0:
        final_price = df_bt.iloc[-1]['Close']
        pnl = (final_price - entry_price) * position
        balance += pnl
        trade_pnls.append(pnl)

        win_flag = pnl > 0
        if win_flag:
            win_count += 1

        trade_log.append({
            'pnl': pnl,
            'win': win_flag,
            'proba': proba_at_trade,
            'market_regime': current_regime,
            'entry_score': entry_score
        })

        equity_curve[-1] = balance

    # ====================== #
    # 指標
    # ====================== #
    win_rate = (win_count / trade_count * 100) if trade_count > 0 else 0
    total_return = (balance / initial_balance - 1) * 100

    equity = np.array(equity_curve)

    # ドローダウン
    peak = np.maximum.accumulate(equity)
    dd = (equity - peak) / peak
    max_dd = dd.min()

    # シャープ（簡易）
    returns = np.diff(equity) / equity[:-1]
    sharpe = np.mean(returns) / (np.std(returns) + 1e-9) * np.sqrt(252)

    # プロフィットファクター
    gains = returns[returns > 0].sum()
    losses = -returns[returns < 0].sum()
    pf = gains / (losses + 1e-9)

    expectancy = np.mean(returns)

    results = {
        "Final Balance": balance,
        "Total Return": total_return,
        "Total Trades": trade_count,
        "Win Rate (%)": win_rate,
        "PF": pf,
        "Expectancy": expectancy,
        "Max DD": max_dd,
        "Sharpe": sharpe
    }

    return results, equity_curve, trade_pnls, trade_log

In [6]:
# ====================== #
# 評価関数 (Updated to include Sharpe Ratio explicitly)
# ====================== #
def calc_metrics(equity, trade_log): # Added trade_log parameter
    equity = np.array(equity)

    if len(equity) < 2:
        return {
            "PF": 0.0, "Expectancy": 0.0, "Max DD": 0.0, "Sharpe": 0.0,
            "Sortino": 0.0, "Stability": 0.0, "Avg_Win_PNL": 0.0,
            "Avg_Loss_PNL": 0.0, "Profit_Efficiency": 0.0
        }

    returns = np.diff(equity) / (equity[:-1] + 1e-9)

    # PF
    gains = returns[returns > 0].sum()
    losses = -returns[returns < 0].sum() # Note: losses here are positive sum of negative returns

    pf = gains / (losses + 1e-9)

    # Expectancy
    expectancy = returns.mean()

    # Max DD
    peak = np.maximum.accumulate(equity)
    dd = (equity - peak) / (peak + 1e-9) # Use (peak + 1e-9) to avoid division by zero
    max_dd = dd.min()

    # Stability (inverse of volatility)
    vol = returns.std() + 1e-9
    stability = 1 / vol

    # Sharpe
    sharpe = expectancy / vol * np.sqrt(252)

    # Sortino (assuming risk-free rate is 0)
    downside = returns[returns < 0]

    if len(downside) < 3: # Need at least 3 values for meaningful std, or more for robust estimation
        sortino = 0.0   # Set to 0 if not enough data for downside deviation
    else:
        sortino = np.mean(returns) / (np.std(downside) + 1e-9) # Use np.std on downside only

    # New calculations from trade_log
    avg_win_pnl = 0.0
    avg_loss_pnl = 0.0
    profit_efficiency = 0.0

    if trade_log: # Only calculate if trade_log is not empty
        df_trade_log = pd.DataFrame(trade_log)
        winning_trades = df_trade_log[df_trade_log['win']]
        losing_trades = df_trade_log[~df_trade_log['win']]

        num_winning_trades = len(winning_trades)
        num_losing_trades = len(losing_trades)

        if num_winning_trades > 0:
            total_profit_of_winning_trades = winning_trades['pnl'].sum()
            avg_win_pnl = total_profit_of_winning_trades / num_winning_trades

        if num_losing_trades > 0:
            total_loss_of_losing_trades = losing_trades['pnl'].sum()
            avg_loss_pnl = abs(total_loss_of_losing_trades) / num_losing_trades

        if avg_loss_pnl > 1e-9: # Avoid division by zero
            profit_efficiency = avg_win_pnl / avg_loss_pnl

    return {
        "PF": pf,
        "Expectancy": expectancy,
        "Max DD": max_dd,
        "Sharpe": sharpe,
        "Sortino": sortino,
        "Stability": stability,
        "Avg_Win_PNL": avg_win_pnl,
        "Avg_Loss_PNL": avg_loss_pnl,
        "Profit_Efficiency": profit_efficiency
    }

In [7]:
# 戦略A・Bのバックテストと結果表示 ( superseded as per user request to unify to Optuna )

In [8]:
# Equity Curveの描画 ( superseded as per user request to unify to Optuna )

In [9]:
try:
    import optuna
except ImportError:
    %pip install optuna
    import optuna

print("Optuna has been successfully imported or installed.")

Optuna has been successfully imported or installed.


In [10]:
# ユーザーからの指摘により、objective関数の重複を避けるため、このセルのobjective関数は削除されました。
# 最新のobjective関数はセル `Bf3eamHoEQfk` に定義されています。


In [11]:
import numpy as np
import pandas as pd
import random
import json

from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE


# ====================== #
# パラメータロード
# ====================== #
def load_params(path="best_params.json"):
    with open(path, "r") as f:
        raw = json.load(f)

    if "params" not in raw:
        raise ValueError("best_params.json missing 'params' key")

    return raw["params"]


# ====================== #
# Optuna Objective (Walk-Forward Analysis Base)
# ====================== #
def objective(trial, X_full, y_full, df_encoded_full, df_original_full):

    def calc_metrics_local(equity, trade_log):
        equity = np.array(equity)

        if len(equity) < 2:
            return 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0 # Sharpe, PF, MaxDD, Stability, Sortino, Expectancy, Avg_Win_PNL, Profit_Efficiency

        returns = np.diff(equity) / (equity[:-1] + 1e-9)

        if returns.size == 0:
            return 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0

        pf = returns[returns > 0].sum() / (-(returns[returns < 0].sum()) + 1e-9)
        vol = returns.std() + 1e-9

        sharpe = (returns.mean() / vol) * np.sqrt(252)
        stability = 1 / vol if vol != 0 else 0.0

        peak = np.maximum.accumulate(equity)
        max_dd = ((equity - peak) / (peak + 1e-9)).min()

        # Sortino (assuming risk-free rate is 0)
        downside = returns[returns < 0]
        if len(downside) < 3:
            sortino = 0.0
        else:
            sortino = np.mean(returns) / (np.std(downside) + 1e-9)

        # New calculations from trade_log
        avg_win_pnl = 0.0
        profit_efficiency = 0.0

        if trade_log: # Only calculate if trade_log is not empty
            df_trade_log = pd.DataFrame(trade_log)
            winning_trades = df_trade_log[df_trade_log['win']]
            losing_trades = df_trade_log[~df_trade_log['win']]

            num_winning_trades = len(winning_trades)
            num_losing_trades = len(losing_trades)

            if num_winning_trades > 0:
                total_profit_of_winning_trades = winning_trades['pnl'].sum()
                avg_win_pnl = total_profit_of_winning_trades / num_winning_trades

            avg_loss_pnl = 0.0
            if num_losing_trades > 0:
                total_loss_of_losing_trades = losing_trades['pnl'].sum()
                avg_loss_pnl = abs(total_loss_of_losing_trades) / num_losing_trades

            if avg_loss_pnl > 1e-9: # Avoid division by zero
                profit_efficiency = avg_win_pnl / avg_loss_pnl

        return sharpe, pf, max_dd, stability, sortino, returns.mean(), avg_win_pnl, profit_efficiency


    seed = 42

    np.random.seed(seed)
    random.seed(seed)

    model_params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma': trial.suggest_float('gamma', 0.0, 0.5),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 0.5),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.5, 2.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10)
    }

    strategy_params = {
        'w_trend': trial.suggest_float('w_trend', 1.5, 3.0),
        'w_vol': trial.suggest_float('w_vol', 0.0, 1.0),
        'w_mom': trial.suggest_float('w_mom', 0.0, 2.5),

        'adx_th': trial.suggest_float('adx_th', 22.0, 30.0),
        'use_slope': trial.suggest_categorical('use_slope', [True, False]),

        'proba_th': trial.suggest_float('proba_th', 0.18, 0.35),

        'proba_th_uptrend_adj': trial.suggest_float('proba_th_uptrend_adj', -0.2, 0.2),
        'proba_th_downtrend_adj': trial.suggest_float('proba_th_downtrend_adj', -0.2, 0.2),
        'proba_th_sideways_adj': trial.suggest_float('proba_th_sideways_adj', -0.2, 0.2),

        'downtrend_momentum_penalty_factor': trial.suggest_float('downtrend_momentum_penalty_factor', 0.1, 1.0),

        'trend_strength_min_abs': trial.suggest_float('trend_strength_min_abs', 3.5, 6.5),

        'proba_filter_type': trial.suggest_categorical('proba_filter_type', ['threshold', 'quantile']),

        'aggressive_size': trial.suggest_float('aggressive_size', 1.8, 2.3),
        'conservative_size': trial.suggest_float('conservative_size', 0.5, 2.0),
        'trend_strength_regime_th': trial.suggest_float('trend_strength_regime_th', 3.0, 10.0),
        'low_trend_strength_cut_off_th': trial.suggest_float('low_trend_strength_cut_off_th', 2.5, 5.0),

        # New parameters for win-maximizing strategy
        'atr_tp_multiplier': trial.suggest_float('atr_tp_multiplier', 2.0, 6.0),
        'atr_sl_multiplier': trial.suggest_float('atr_sl_multiplier', 0.5, 2.5),
        'trail_stop': True,
        'trail_ratio': trial.suggest_float('trail_ratio', 0.3, 0.9),
        'min_hold_bars': trial.suggest_int('min_hold_bars', 2, 10)
    }

    # Walk-forward setup
    initial_train_period_years = 3
    test_period_years = 1

    min_date = df_original_full.index.min()
    max_date = df_original_full.index.max()

    current_train_end = min_date + pd.DateOffset(years=initial_train_period_years)

    fold_results = []

    fold_count = 0
    while current_train_end < max_date - pd.DateOffset(days=30):
        fold_count += 1

        # Train Period Data
        X_train_wf = X_full[(X_full.index >= min_date) & (X_full.index <= current_train_end)]
        y_train_wf = y_full[(y_full.index >= min_date) & (y_full.index <= current_train_end)]
        df_encoded_train_wf = df_encoded_full[(df_encoded_full.index >= min_date) & (df_encoded_full.index <= current_train_end)]

        # Test Period Data
        test_start = current_train_end + pd.DateOffset(days=1)
        test_end = test_start + pd.DateOffset(years=test_period_years) - pd.DateOffset(days=1)
        if test_end > max_date:
            test_end = max_date

        X_test_wf = X_full[(X_full.index >= test_start) & (X_full.index <= test_end)]
        y_test_wf = y_full[(y_full.index >= test_start) & (y_full.index <= test_end)] # Not directly used for model training/signal gen, but for consistency
        df_encoded_test_wf = df_encoded_full[(df_encoded_full.index >= test_start) & (df_encoded_full.index <= test_end)]

        if X_test_wf.empty or len(X_test_wf) < 2: # Testデータが少なすぎる場合は終了
            break

        # --- Model training and prediction (using the current fold's data) ---
        smote = SMOTE(random_state=seed)
        X_res, y_res = smote.fit_resample(X_train_wf, y_train_wf)

        scaler = StandardScaler()
        X_res = scaler.fit_transform(X_res)
        X_test_scaled = scaler.transform(X_test_wf)

        model = XGBClassifier(
            **model_params,
            random_state=seed,
            eval_metric='logloss',
            n_jobs=-1
        )
        model.fit(X_res, y_res)

        proba = model.predict_proba(X_test_scaled)[:, 1]
        proba_series = pd.Series(proba, index=X_test_wf.index)

        # --- Signal generation (using strategy_params from trial) ---
        trend_strength_series = df_encoded_test_wf["TREND_STRENGTH"]
        volatility_series = df_encoded_test_wf["Volatility_Short"]
        momentum_series = df_encoded_test_wf["RET_5"]
        adx_series = df_encoded_test_wf["ADX"]

        market_regime_test_wf_str = pd.Series('sideways', index=X_test_wf.index)
        if 'Market_Regime_uptrend' in df_encoded_test_wf.columns:
            uptrend_mask = (df_encoded_test_wf['Market_Regime_uptrend'] == 1)
            market_regime_test_wf_str[uptrend_mask] = 'uptrend'
        if 'Market_Regime_downtrend' in df_encoded_test_wf.columns:
            downtrend_mask = (df_encoded_test_wf['Market_Regime_downtrend'] == 1)
            market_regime_test_wf_str[downtrend_mask] = 'downtrend'

        mom_adj = momentum_series.copy()
        downtrend_mask_regime = (market_regime_test_wf_str == 'downtrend')
        mom_adj[downtrend_mask_regime] *= strategy_params.get('downtrend_momentum_penalty_factor', 1.0)

        entry_score_series = (
            proba_series
            + strategy_params.get('w_trend', 0.0) * trend_strength_series.abs()
            + strategy_params.get('w_mom', 0.0) * mom_adj
            - strategy_params.get('w_vol', 0.0) * volatility_series
        )

        adx_ma = adx_series.rolling(20, min_periods=1).mean()
        cond_trend = pd.Series(True, index=X_test_wf.index)
        if strategy_params.get('use_slope', False):
            adx_slope = adx_series.diff().rolling(5, min_periods=1).mean()
            cond_trend = (
                (adx_ma > strategy_params.get('adx_th', 20.0)) &
                (adx_slope > 0) &
                (trend_strength_series.abs() > strategy_params.get('trend_strength_min_abs', 0.0))
            )
        else:
            cond_trend = (
                (adx_ma > strategy_params.get('adx_th', 20.0)) &
                (trend_strength_series.abs() > strategy_params.get('trend_strength_min_abs', 0.0))
            )

        cond_proba = pd.Series(True, index=X_test_wf.index)
        if strategy_params.get('proba_filter_type', 'threshold') == 'threshold':
            base_proba_th = strategy_params.get('proba_th', 0.5)
            adjusted_proba_th_arr = np.full(len(proba_series), base_proba_th)

            is_uptrend = (market_regime_test_wf_str == 'uptrend').values
            adjusted_proba_th_arr = np.where(is_uptrend, base_proba_th + strategy_params.get('proba_th_uptrend_adj', 0.0), adjusted_proba_th_arr)
            is_downtrend = (market_regime_test_wf_str == 'downtrend').values
            # Corrected logic: base_proba_th is already the 'base', just add the adjustment
            adjusted_proba_th_arr = np.where(is_downtrend, base_proba_th + strategy_params.get('proba_th_downtrend_adj', 0.0), adjusted_proba_th_arr)
            is_sideways = (market_regime_test_wf_str == 'sideways').values
            adjusted_proba_th_arr = np.where(is_sideways, base_proba_th + strategy_params.get('proba_th_sideways_adj', 0.0), adjusted_proba_th_arr)

            cond_proba = proba_series > adjusted_proba_th_arr
        else: # 'quantile'
            cond_proba = proba_series > proba_series.quantile(strategy_params.get('proba_th', 0.5))

        base_signal = cond_trend & cond_proba

        signal_multipliers = pd.Series(0.0, index=X_test_wf.index)
        if not entry_score_series.loc[base_signal].empty:
            strength_on_signal_days = entry_score_series.loc[base_signal].rank(pct=True)

            base_size = pd.Series(strategy_params.get('conservative_size', 1.0), index=X_test_wf.index)
            aggressive_mask_days = (trend_strength_series.abs() > strategy_params.get('trend_strength_regime_th', 5.0))
            base_size[aggressive_mask_days] = strategy_params.get('aggressive_size', 1.5)

            signal_multipliers.loc[base_signal] = strength_on_signal_days * base_size.loc[base_signal]

        cut_off_mask = (trend_strength_series.abs() < strategy_params.get('low_trend_strength_cut_off_th', 0.0))
        signal_multipliers[cut_off_mask] = 0.0

        df_trade_info_fold = pd.DataFrame({
            "proba": proba_series,
            "entry_score": entry_score_series,
            "Market_Regime": market_regime_test_wf_str
        })

        results, equity, _, trade_log_fold = run_backtest(
            df_original_full,
            X_test_wf,
            signal_multipliers,
            df_trade_info_fold,
            atr_tp_multiplier=strategy_params['atr_tp_multiplier'],
            atr_sl_multiplier=strategy_params['atr_sl_multiplier'],
            trail_stop=strategy_params['trail_stop'],
            trail_ratio=strategy_params['trail_ratio'],
            min_hold_bars=strategy_params['min_hold_bars']
        )

        sharpe, pf, max_dd, stab, sortino, expectancy, avg_win_pnl, profit_efficiency = calc_metrics_local(equity, trade_log_fold)
        total_trades_fold = results["Total Trades"]

        # Immediate rejection criteria
        if sharpe < 0 or pf < 1.1 or abs(max_dd) > 0.1 or total_trades_fold < 1 or total_trades_fold > 15:
            return -float('inf')

        fold_results.append({
            "sharpe": sharpe,
            "return": results["Total Return"] / 100, # Convert percentage to decimal
            "max_dd": max_dd,
            "trades": total_trades_fold,
            "pf": pf,
            "avg_win_pnl": avg_win_pnl,
            "profit_efficiency": profit_efficiency
        })

        current_train_end = current_train_end + pd.DateOffset(years=test_period_years)

    if not fold_results: # If no folds ran or all were rejected
        return -float('inf')

    # Aggregate and score
    mean_sharpe = np.mean([f["sharpe"] for f in fold_results])
    min_sharpe = np.min([f["sharpe"] for f in fold_results])
    mean_return = np.mean([f["return"] for f in fold_results])
    mean_dd = np.mean([f["max_dd"] for f in fold_results])
    std_sharpe = np.std([f["sharpe"] for f in fold_results])
    mean_pf = np.mean([f["pf"] for f in fold_results])
    mean_avg_win_pnl = np.mean([f["avg_win_pnl"] for f in fold_results])
    mean_profit_efficiency = np.mean([f["profit_efficiency"] for f in fold_results])

    score = (
        1.5 * mean_sharpe
        + 1.2 * mean_return
        - 1.0 * abs(mean_dd)
        - 0.5 * std_sharpe
        + 1.0 * mean_pf
        + 0.8 * mean_avg_win_pnl
        + 0.7 * mean_profit_efficiency
    )

    # Penalize if insufficient trades overall (user guidance)
    total_trades_overall = sum(f["trades"] for f in fold_results)
    # if total_trades_overall < 15:
    #     score -= (15 - total_trades_overall) * 0.3
    # elif total_trades_overall < 60:
    #     score -= 0.3

    trial.set_user_attr("fold_metrics", fold_results)
    trial.set_user_attr("total_trades_overall", total_trades_overall)

    return score

In [ ]:
import optuna
import os
import json
import numpy as np

# ====================== #
# DBリセット（必要なら）
# ====================== #
db_path = "optuna_study.db"

if os.path.exists(db_path):
    os.remove(db_path)
    print("old study removed")

# ====================== #
# study作成
# ====================== #
study = optuna.create_study(
    direction="maximize",
    study_name="trading_optuna",
    storage=f"sqlite:///{db_path}",
    load_if_exists=True,

    sampler=optuna.samplers.TPESampler(
        seed=42,
        multivariate=True,
        group=True
    ),

    pruner=optuna.pruners.HyperbandPruner(
        min_resource=5,
        max_resource=100,
        reduction_factor=3
    )
)

# ====================== #
# 最適化実行
# ====================== #
# Pass X, y, df_encoded, and df (original DataFrame) to the objective function
study.optimize(lambda trial: objective(trial, X, y, df_encoded, df), n_trials=300)

# ====================== #
# ベスト取得
# ====================== #
best_trial = study.best_trial
best_params = best_trial.params
best_score = best_trial.value

print("\n===== BEST RESULT === ==")
print("score:", best_score)

for k, v in best_params.items():
    print(k, ":", v)

print("trials:", len(study.trials))

# ====================== #
# 保存
# ====================== #
with open("best_params.json", "w") as f:
    json.dump({
        "score": best_score,
        "params": best_params,
        "fold_metrics": best_trial.user_attrs.get("fold_metrics", None),
        "total_trades_overall": best_trial.user_attrs.get("total_trades_overall", None)
    }, f, indent=4)

old study removed


/tmp/ipykernel_59200/2283444199.py:24: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  sampler=optuna.samplers.TPESampler(
/tmp/ipykernel_59200/2283444199.py:24: ExperimentalWarning: Argument ``group`` is an experimental feature. The interface can change in the future.
  sampler=optuna.samplers.TPESampler(
[I 2026-05-04 21:17:12,019] A new study created in RDB with name: trading_optuna
[I 2026-05-04 21:17:18,302] Trial 0 finished with value: -inf and parameters: {'n_estimators': 250, 'max_depth': 12, 'learning_rate': 0.08960785365368121, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.6624074561769746, 'gamma': 0.07799726016810132, 'reg_alpha': 0.02904180608409973, 'reg_lambda': 1.7992642186624028, 'min_child_weight': 7, 'w_trend': 2.5621088666940683, 'w_vol': 0.020584494295802447, 'w_mom': 2.4247746304049858, 'adx_th': 28.659541126403376, 'use_slope': True, 'proba_th': 0.21117876667508373, 'proba_th_uptrend_ad

In [ ]:
import json
import numpy as np
import os # Ensure os is imported for os.listdir

# Check if model_params is defined, if not, try to get it from best_params (which should exist after Optuna)
# Load all best parameters (model and strategy) from best_params.json
# Set default values for strategy parameters to ensure robustness
# Always attempt to load parameters from file to ensure latest are used
print("Attempting to load parameters from best_params.json...")
try:
    with open('best_params.json', 'r') as f:
        loaded_best_params = json.load(f)
    actual_params = loaded_best_params.get('params', {})

    model_params = {
        k: actual_params[k]
        for k in [
            'n_estimators', 'max_depth', 'learning_rate',
            'subsample', 'colsample_bytree',
            'gamma', 'reg_alpha', 'reg_lambda', 'min_child_weight'
        ] if k in actual_params
    }
    print("DEBUG: model_params after loading:", model_params)

    # Define strategy_params with default values
    strategy_params = {
        'w_trend': actual_params.get('w_trend', 0.0),
        'w_vol': actual_params.get('w_vol', 0.0),
        'w_mom': actual_params.get('w_mom', 0.0),
        'adx_th': actual_params.get('adx_th', 20.0),
        'use_slope': actual_params.get('use_slope', False),
        'proba_th': actual_params.get('proba_th', 0.5),
        'proba_th_uptrend_adj': actual_params.get('proba_th_uptrend_adj', 0.0),
        'proba_th_downtrend_adj': actual_params.get('proba_th_downtrend_adj', 0.0),
        'proba_th_sideways_adj': actual_params.get('proba_th_sideways_adj', 0.0),
        'downtrend_momentum_penalty_factor': actual_params.get('downtrend_momentum_penalty_factor', 1.0),
        'trend_strength_min_abs': actual_params.get('trend_strength_min_abs', 0.0),
        'proba_filter_type': actual_params.get('proba_filter_type', 'threshold'),
        'aggressive_size': actual_params.get('aggressive_size', 1.5),
        'conservative_size': actual_params.get('conservative_size', 1.0),
        'trend_strength_regime_th': actual_params.get('trend_strength_regime_th', 5.0),
        'low_trend_strength_cut_off_th': actual_params.get('low_trend_strength_cut_off_th', 0.0)
    }
    print("DEBUG: strategy_params after loading:", strategy_params)
    print("Model and Strategy parameters loaded successfully from best_params.json.")

except json.JSONDecodeError:
    print("Error: best_params.json is corrupted. Using default model and strategy parameters.")
    model_params = {}
    strategy_params = {}
except Exception as e:
    print(f"Error loading best_params.json: {e}. Using default model and strategy parameters.")
    model_params = {}
    strategy_params = {}

# Initialize model with the newly loaded parameters
model = XGBClassifier(
    **model_params,
    random_state=42,
    eval_metric='logloss',
    n_jobs=-1
)


In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
import pandas as pd
import numpy as np
import json
from sklearn.preprocessing import StandardScaler

# ====================== #
# 指標関数 (Updated to include Sharpe Ratio explicitly)
# ====================== #

def calc_metrics(equity):
    equity = np.array(equity)

    if len(equity) < 2:
        return { "PF": 0.0, "Expectancy": 0.0, "Max DD": 0.0, "Sharpe": 0.0, "Sortino": 0.0, "Stability": 0.0 }

    returns = np.diff(equity) / (equity[:-1] + 1e-9)

    # PF
    gains = returns[returns > 0].sum()
    losses = -returns[returns < 0].sum()
    pf = gains / (losses + 1e-9)

    # Expectancy
    expectancy = returns.mean()

    # Max DD
    peak = np.maximum.accumulate(equity)
    dd = (equity - peak) / (peak + 1e-9) # Use (peak + 1e-9) to avoid division by zero
    max_dd = dd.min()

    # Stability (inverse of volatility)
    vol = returns.std() + 1e-9
    stability = 1 / vol

    # Sharpe
    sharpe = expectancy / vol * np.sqrt(252)

    # Sortino (assuming risk-free rate is 0)
    downside = returns[returns < 0]

    if len(downside) < 3: # Need at least 3 values for meaningful std, or more for robust estimation
        sortino = 0.0   # Set to 0 if not enough data for downside deviation
    else:
        sortino = np.mean(returns) / (np.std(downside) + 1e-9)

    return {
        "PF": pf,
        "Expectancy": expectancy,
        "Max DD": max_dd,
        "Sharpe": sharpe,
        "Sortino": sortino,
        "Stability": stability
    }

# ====================== #
# CV 評価関数
# ====================== #

def evaluate_strategy_cv(base_strategy_params, X, y, df_encoded, df_original, model_params, filter_to_exclude=None):
    """
    Runs Cross-Validation for the trading strategy with specified filter exclusions.

    Args:
        base_strategy_params (dict): The base strategy parameters.
        X (pd.DataFrame): Feature DataFrame.
        y (pd.Series): Target Series.
        df_encoded (pd.DataFrame): DataFrame with all features, including encoded market regimes.
        df_original (pd.DataFrame): Original DataFrame (used by run_backtest for Close, ATR, etc.).
        filter_to_exclude (str, optional): Which filter to exclude ('score', 'trend', 'proba'). Defaults to None.

    Returns:
        pd.DataFrame: Summary of CV results.
        list: List of equity curves for each fold.
    """

    tscv = TimeSeriesSplit(n_splits=10)
    results_list = []
    equity_curves = []
    seed = 42 # For reproducibility within CV folds

    for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):

        X_train_cv, X_test_cv = X.iloc[train_idx], X.iloc[test_idx]
        y_train_cv, y_test_cv = y.iloc[train_idx], y.iloc[test_idx]

        # SMOTE
        smote = SMOTE(random_state=seed)
        X_train_resampled_array, y_train_res = smote.fit_resample(X_train_cv, y_train_cv)
        X_train_res = pd.DataFrame(X_train_resampled_array, columns=X_train_cv.columns, index=y_train_res.index)

        # Scaling
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_res)
        X_test_scaled = scaler.transform(X_test_cv)

        # Model
        model = XGBClassifier(
            **model_params,
            random_state=seed,
            eval_metric='logloss',
            n_jobs=-1
        )
        model.fit(X_train_scaled, y_train_res)

        # Prediction
        proba = model.predict_proba(X_test_scaled)[:, 1]
        proba_series = pd.Series(proba, index=X_test_cv.index)

        # Access features directly from df_encoded using the test set index.
        trend_strength_series = df_encoded.loc[X_test_cv.index, "TREND_STRENGTH"]
        volatility_series = df_encoded.loc[X_test_cv.index, "Volatility_Short"]
        momentum_series = df_encoded.loc[X_test_cv.index, "RET_5"]
        adx_series = df_encoded.loc[X_test_cv.index, "ADX"]

        # Strategy parameters for this run (deep copy to avoid modifying original)
        strategy_params = base_strategy_params.copy()

        momentum_term_series = momentum_series.copy()
        if 'Market_Regime_downtrend' in df_encoded.columns:
            is_downtrend_regime = df_encoded.loc[X_test_cv.index, 'Market_Regime_downtrend'] == 1
            momentum_term_series[is_downtrend_regime] *= strategy_params['downtrend_momentum_penalty_factor']

        # Calculate Entry Score as a separate Series
        entry_score_series = (
            proba_series
            + strategy_params['w_trend'] * trend_strength_series.abs()
            + strategy_params['w_mom'] * momentum_term_series
            - strategy_params['w_vol'] * volatility_series
        )

        # Trend Filter (using adx_series and trend_strength_series)
        adx_ma = adx_series.rolling(20, min_periods=1).mean()
        adx_slope = adx_series.diff()
        adx_slope_mean = adx_slope.rolling(5, min_periods=1).mean()

        if strategy_params['use_slope']:
            cond_trend = (
                (adx_ma > strategy_params['adx_th']) &
                (adx_slope_mean > 0) &
                (trend_strength_series.abs() > strategy_params['trend_strength_min_abs'])
            )
        else:
            cond_trend = (
                (adx_ma > strategy_params['adx_th']) &
                (trend_strength_series.abs() > strategy_params['trend_strength_min_abs'])
            )


        # Proba Filter (using proba_series and market regime from df_encoded)
        adjusted_proba_th_arr = np.full(len(proba_series), strategy_params['proba_th'])
        market_regime_test_cv_str = pd.Series('sideways', index=X_test_cv.index) # Default to 'sideways'
        if 'Market_Regime_uptrend' in df_encoded.columns:
            uptrend_mask = (df_encoded.loc[X_test_cv.index, 'Market_Regime_uptrend'] == 1)
            market_regime_test_cv_str[uptrend_mask] = 'uptrend'
        if 'Market_Regime_downtrend' in df_encoded.columns:
            downtrend_mask = (df_encoded.loc[X_test_cv.index, 'Market_Regime_downtrend'] == 1)
            market_regime_test_cv_str[downtrend_mask] = 'downtrend'

        if strategy_params['proba_filter_type'] == 'threshold':
            is_uptrend = (market_regime_test_cv_str == 'uptrend').values
            adjusted_proba_th_arr = np.where(is_uptrend, strategy_params['proba_th'] + strategy_params['proba_th_uptrend_adj'], adjusted_proba_th_arr)
            is_downtrend = (market_regime_test_cv_str == 'downtrend').values
            adjusted_proba_th_arr = np.where(is_downtrend, strategy_params['proba_th'] + strategy_params['proba_th_downtrend_adj'], adjusted_proba_th_arr)
            is_sideways = (market_regime_test_cv_str == 'sideways').values
            adjusted_proba_th_arr = np.where(is_sideways, strategy_params['proba_th'] + strategy_params['proba_th_sideways_adj'], adjusted_proba_th_arr)
            cond_proba = proba_series > adjusted_proba_th_arr
        else: # 'quantile'
            cond_proba = proba_series > proba_series.quantile(strategy_params['proba_th'])

        # --- Apply filter exclusions based on filter_to_exclude argument ---
        temp_cond_trend = cond_trend.copy()
        temp_cond_proba = cond_proba.copy()

        if filter_to_exclude == 'trend':
            temp_cond_trend = pd.Series(True, index=X_test_cv.index)
        if filter_to_exclude == 'proba':
            temp_cond_proba = pd.Series(True, index=X_test_cv.index)

        # ① コアフィルター
        base_signal = temp_cond_trend & temp_cond_proba

        # ② ポジションサイズを決定（エントリースコアによる強弱）
        signal_multipliers = pd.Series(0.0, index=X_test_cv.index)

        if filter_to_exclude == 'score':
            # スコアフィルターなしの場合、全てのベースシグナルにデフォルトポジションサイズ1.0を割り当てる
            signal_multipliers.loc[base_signal] = 1.0
        else:
            # base_signalがある日に対して、エントリースコアに応じてポジションサイズを設定
            if not entry_score_series.loc[base_signal].empty:
                strength_on_signal_days = entry_score_series.loc[base_signal].rank(pct=True)
                # Determine base size (conservative or aggressive) based on strategy_params
                base_size = pd.Series(strategy_params.get('conservative_size', 1.0), index=X_test_cv.index)
                if 'trend_strength_regime_th' in strategy_params:
                    aggressive_mask_days = (trend_strength_series.abs() > strategy_params['trend_strength_regime_th'])
                    base_size[aggressive_mask_days] = strategy_params.get('aggressive_size', 1.5)

                signal_multipliers.loc[base_signal] = strength_on_signal_days * base_size.loc[base_signal]

        # Low Trend Strength Cut-off
        if 'low_trend_strength_cut_off_th' in strategy_params:
            cut_off_mask = (trend_strength_series.abs() < strategy_params['low_trend_strength_cut_off_th'])
            signal_multipliers[cut_off_mask] = 0.0

        # Create df_trade_info for the fold (required by run_backtest)
        df_trade_info_fold_data = {
            'proba': proba_series,
            'entry_score': entry_score_series,
            'Market_Regime': market_regime_test_cv_str
        }
        df_trade_info_fold = pd.DataFrame(df_trade_info_fold_data)

        # backtest
        results, equity, _, _ = run_backtest(
            df_original, # バックテスト関数には元の df を渡す (Close, ATR など生のデータが必要なため)
            X_test_cv,
            signal_multipliers,
            df_trade_info_fold
        )

        # Add all metrics
        metrics = calc_metrics(equity)
        results.update(metrics)
        results["Total Signal Days"] = signal_multipliers[signal_multipliers > 0].count()

        results_list.append(results)
        equity_curves.append(equity)

    return pd.DataFrame(results_list), equity_curves


# --- Main script to load params and run CV for different scenarios ---

# Always attempt to load parameters from file to ensure latest are used
print("Attempting to load parameters from best_params.json for CV scenarios...")
try:
    with open("best_params.json", "r") as f:
        loaded_best_params = json.load(f)

    actual_params = loaded_best_params.get('params', {})

    model_params = {
        k: actual_params[k]
        for k in [
            'n_estimators', 'max_depth', 'learning_rate',
            'subsample', 'colsample_bytree',
            'gamma', 'reg_alpha', 'reg_lambda', 'min_child_weight'
        ] if k in actual_params
    }

    base_strategy_params = {
        'w_trend': actual_params.get('w_trend', 0.0),
        'w_vol': actual_params.get('w_vol', 0.0),
        'w_mom': actual_params.get('w_mom', 0.0),
        'adx_th': actual_params.get('adx_th', 20.0),
        'use_slope': actual_params.get('use_slope', False),
        'proba_th': actual_params.get('proba_th', 0.5),
        'proba_th_uptrend_adj': actual_params.get('proba_th_uptrend_adj', 0.0),
        'proba_th_downtrend_adj': actual_params.get('proba_th_downtrend_adj', 0.0),
        'proba_th_sideways_adj': actual_params.get('proba_th_sideways_adj', 0.0),
        'downtrend_momentum_penalty_factor': actual_params.get('downtrend_momentum_penalty_factor', 1.0),
        'trend_strength_min_abs': actual_params.get('trend_strength_min_abs', 0.0),
        'proba_filter_type': actual_params.get('proba_filter_type', 'threshold'),
        'aggressive_size': actual_params.get('aggressive_size', 1.5),
        'conservative_size': actual_params.get('conservative_size', 1.0),
        'trend_strength_regime_th': actual_params.get('trend_strength_regime_th', 5.0),
        'low_trend_strength_cut_off_th': actual_params.get('low_trend_strength_cut_off_th', 0.0)
    }
    print("Parameters loaded successfully from best_params.json.")
except FileNotFoundError:
    print("Error: best_params.json not found. Skipping CV scenarios.")
    model_params = {}
    base_strategy_params = {}
except Exception as e:
    print(f"Error loading parameters: {e}. Skipping CV scenarios.")
    model_params = {}
    base_strategy_params = {}

# Ensure X, y, df_encoded, df are available globally from previous cells
if not all(v is not None for v in [X, y, df_encoded, df, model_params, base_strategy_params]):
    print("Error: Required variables (X, y, df_encoded, df, model_params, base_strategy_params) are not fully defined. Cannot run CV scenarios.")
else:
    # Modified scenarios to only include baseline and no score filter
    scenarios = {
        "ベースライン (全フィルター有効)": None,
        "スコアフィルターなし": "score"
    }

    all_scenario_results = {}

    for scenario_name, filter_excl in scenarios.items():
        print(f"\n===== シナリオ: {scenario_name} ====")
        df_results_scenario, _ = evaluate_strategy_cv(base_strategy_params, X, y, df_encoded, df, model_params, filter_to_exclude=filter_excl)

        # Print results for each fold
        print("\n各フォールドの結果:")
        print(df_results_scenario)

        # Optionally, still print the mean for quick summary
        print("\nCV SUMMARY (平均):")
        print(df_results_scenario.mean())

        all_scenario_results[scenario_name] = df_results_scenario

    print("\n===== 全シナリオ結果の比較 ====")
    for scenario_name, results_df in all_scenario_results.items():
        print(f"\n--- {scenario_name} ---")
        print(results_df[['Total Trades', 'Total Return', 'Sharpe', 'PF', 'Max DD']])


### ウォークフォワード検証の結果概要

In [ ]:
import json
import pandas as pd

try:
    with open("best_params.json", "r") as f:
        best_result_data = json.load(f)

    fold_metrics = best_result_data.get("fold_metrics")
    total_trades_overall = best_result_data.get("total_trades_overall")
    best_score = best_result_data.get("score")

    print("===== Optuna最適化のベストスコア =====")
    print(f"総合スコア: {best_score:.4f}")
    print(f"全フォールドでの総取引回数: {total_trades_overall}")

    if fold_metrics:
        df_fold_results = pd.DataFrame(fold_metrics)
        print("\n===== ウォークフォワード各フォールドのメトリクス =====")
        display(df_fold_results)
        print("\n===== ウォークフォワードメトリクス (平均) =====")
        display(df_fold_results.mean().to_frame().T)
    else:
        print("ウォークフォワードのフォールドメトリクスが見つかりませんでした。")

except FileNotFoundError:
    print("エラー: best_params.json が見つかりません。Optunaの実行を確認してください。")
except Exception as e:
    print(f"エラー発生: {e}")
